In [2]:
import pandas as pd
import numpy as np
from scipy import stats
from itertools import combinations, product
from statsmodels.stats.multitest import multipletests
from scipy.stats import shapiro

# ========== Настройки ==========
# путь к файлу (Excel или CSV)
input_file = r"C:\Users\user\Documents\IMBIT\my_data.xlsx"  # <--- поменяй путь
sheet_name = 0  # или "Sheet1"
use_excel = True  # если False, ожидать CSV

# имена колонок с группами (впрямую в таблице три столбца)
cols = ["group1", "group2", "group3"]  # <--- поменяй на свои имена столбцов

# permutation settings
n_permutations = 10000

# ========== Загружаем данные ==========
if use_excel:
    df = pd.read_excel(input_file, sheet_name=sheet_name)
else:
    df = pd.read_csv(input_file)

# вытаскиваем колонки и очищаем от нечисловых/NaN
groups = {}
for c in cols:
    if c not in df.columns:
        raise KeyError(f"Колонки '{c}' нет в таблице. Доступные: {df.columns.tolist()}")
    arr = pd.to_numeric(df[c], errors="coerce")
    arr = arr.dropna().values
    groups[c] = arr

# информативный вывод размеров
for name, arr in groups.items():
    print(f"{name}: n={len(arr)} (median={np.median(arr):.3g}, IQR={np.percentile(arr,75)-np.percentile(arr,25):.3g})")
    
    
for name, arr in groups.items():
    if len(arr) < 3:
        print(f"{name}: слишком мало данных для Shapiro-Wilk (нужно ≥3)")
        continue
    try:
        stat, p = shapiro(arr)
        normality = "нормальное" if p > 0.05 else "ненормальное"
        print(f"{name}: stat={stat:.3f}, p={p:.5f} → {normality}")
    except Exception as e:
        print(f"{name}: Shapiro-Wilk не удалось посчитать ({e})")
        
        
# ========== Kruskal-Wallis ==========
arrays = [groups[c] for c in cols]
kw_stat, kw_p = stats.kruskal(*arrays)
n_total = sum(len(a) for a in arrays)
k = len(arrays)

# epsilon-squared for Kruskal-Wallis: (H - k + 1)/(n - k)
epsilon_sq = (kw_stat - k + 1) / (n_total - k) if n_total > k else np.nan

print("\nKruskal-Wallis:")
print(f"H = {kw_stat:.4f}, p = {kw_p:.5f}, epsilon^2 = {epsilon_sq:.4f}")

# ========== Pairwise comparisons ==========
pairs = list(combinations(cols, 2))
pair_results = []
u_stats = []
p_vals = []

# Function Cliff's delta
def cliffs_delta(x, y):
    n, m = len(x), len(y)
    more = 0
    less = 0
    for xi in x:
        for yi in y:
            if xi > yi:
                more += 1
            elif xi < yi:
                less += 1
    delta = (more - less) / (n * m)
    return delta

# permutation test for two-sample difference in medians (or ranks)
def permutation_test(x, y, n_perm=5000, stat_func=lambda a,b: np.median(a)-np.median(b)):
    rng = np.random.default_rng()
    obs = stat_func(x, y)
    pooled = np.concatenate([x, y])
    count = 0
    perm_stats = []
    for _ in range(n_perm):
        rng.shuffle(pooled)
        a = pooled[:len(x)]
        b = pooled[len(x):]
        s = stat_func(a, b)
        perm_stats.append(s)
        if abs(s) >= abs(obs):
            count += 1
    p = (count + 1) / (n_perm + 1)
    return obs, p, np.array(perm_stats)

def significance_stars(p, alpha=0.05, m=3):
    """
    Присваивает звездочки значимости с поправкой Бонферрони
    p : float (p-value уже скорректированная по Бонферрони)
    alpha : float (обычно 0.05)
    m : int (число сравнений, нужно для контроля порогов)
    """
    # вычисляем новые пороги
    alpha1 = alpha / m
    alpha2 = 0.01 / m
    alpha3 = 0.001 / m

    if p < alpha3:
        return "***"
    elif p < alpha2:
        return "**"
    elif p < alpha1:
        return "*"
    else:
        return "n.s."

for a,b in pairs:
    x = groups[a]
    y = groups[b]
    # Mann-Whitney U (two-sided)
    try:
        u_stat, p_mw = stats.mannwhitneyu(x, y, alternative="two-sided")
    except Exception as e:
        u_stat, p_mw = np.nan, np.nan
    # permutation test on median difference (robust for small n)
    obs_med, p_perm, perm_dist = permutation_test(x, y, n_perm=n_permutations, stat_func=lambda aa,bb: np.median(aa)-np.median(bb))
    # cliff's delta
    delta = cliffs_delta(x, y)
    pair_results.append({
        "pair": f"{a} vs {b}",
        "n_a": len(x),
        "n_b": len(y),
        "MannU": u_stat,
        "p_MannWhitney": p_mw,
        "median_diff": obs_med,
        "p_permutation_median": p_perm,
        "cliffs_delta": delta
    })
    p_vals.append(p_mw if not np.isnan(p_mw) else p_perm)  # for multiplicity correction choose MW p if available else permutation p

# multiple testing correction (Bonferroni and BH)
p_array = np.array(p_vals)
reject_bonf, p_bonf, _, _ = multipletests(p_array, alpha=0.05, method="bonferroni")
reject_bh, p_bh, _, _ = multipletests(p_array, alpha=0.05, method="fdr_bh")

# attach corrected p to results
for i, d in enumerate(pair_results):
    d["p_bonferroni"] = p_bonf[i]
    d["reject_bonferroni"] = bool(reject_bonf[i])
    d["p_BH"] = p_bh[i]
    d["reject_BH"] = bool(reject_bh[i])

# ========== Outputs ==========
res_df = pd.DataFrame(pair_results)
print("\nPairwise results:")
print(res_df)

# Применяем к таблице
res_df["significance"] = res_df["p_bonferroni"].apply(significance_stars)
display(res_df) 
# Save results
#res_df.to_excel(r"C:\Users\Lisa\Documents\pairwise_results.xlsx", index=False)
#print("\nРезультаты парных сравнений сохранены в pairwise_results.xlsx")


group1: n=8 (median=13.5, IQR=6.18)
group2: n=11 (median=18.2, IQR=6.85)
group3: n=7 (median=16.4, IQR=6.4)
group1: stat=0.971, p=0.90892 → нормальное
group2: stat=0.949, p=0.62885 → нормальное
group3: stat=0.856, p=0.13815 → нормальное

Kruskal-Wallis:
H = 3.7375, p = 0.15432, epsilon^2 = 0.0755

Pairwise results:
               pair  n_a  n_b  MannU  p_MannWhitney  median_diff  \
0  group1 vs group2    8   11   22.0       0.075848         -4.7   
1  group1 vs group3    8    7   15.5       0.164537         -2.9   
2  group2 vs group3   11    7   41.5       0.820786          1.8   

   p_permutation_median  cliffs_delta  p_bonferroni  reject_bonferroni  \
0              0.056594     -0.500000      0.227543              False   
1              0.398460     -0.446429      0.493612              False   
2              1.000000      0.077922      1.000000              False   

       p_BH  reject_BH  
0  0.227543      False  
1  0.246806      False  
2  0.820786      False  


,pair,n_a,n_b,MannU,p_MannWhitney,median_diff,p_permutation_median,cliffs_delta,p_bonferroni,reject_bonferroni,p_BH,reject_BH,significance
0,group1 vs group2,8,11,22.0,0.075848,-4.7,0.056594,-0.500000,0.227543,False,0.227543,False,n.s.
1,group1 vs group3,8,7,15.5,0.164537,-2.9,0.398460,-0.446429,0.493612,False,0.246806,False,n.s.
2,group2 vs group3,11,7,41.5,0.820786,1.8,1.000000,0.077922,1.000000,False,0.820786,False,n.s.


In [4]:
conda install -c conda-forge statsmodels

Solving environment: ...working... done

## Package Plan ##

  environment location: C:\Users\user\anaconda3\envs\ElectroProp

  added / updated specs:
    - statsmodels


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    ca-certificates-2025.8.3   |       h4c7d964_0         151 KB  conda-forge
    intel-openmp-2025.2.0      |     h57928b3_757        21.4 MB  conda-forge
    libblas-3.9.0              |  28_h576b46c_mkl         3.6 MB  conda-forge
    libcblas-3.9.0             |  28_h7ad3364_mkl         3.6 MB  conda-forge
    liblapack-3.9.0            |  28_hacfb0e4_mkl         3.6 MB  conda-forge
    m2w64-gcc-libgfortran-5.3.0|                6         342 KB  conda-forge
    m2w64-gcc-libs-5.3.0       |                7         520 KB  conda-forge
    m2w64-gcc-libs-core-5.3.0  |                7         214 KB  conda-forge
    m2w64-gmp-6.1.0            |                2         72



==> WARNING: A newer version of conda exists. <==
  current version: 4.10.1
  latest version: 25.7.0

Please update conda by running

    $ conda update -n base -c defaults conda


